# Experiment 10: Baseline vs Human (3 Students)

This notebook runs the BASELINE Curriculum-Aware prompt (WITHOUT the student mental model) on the identical 50 problem subset for the 3 human validation students (10155, 14475, 14476). 

We then compare the baseline LLM's tags against Human raters and the Enriched LLM (curriculum + mental model) to determine if adding the mental model significantly improves human alignment (kappa).

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Dict, List, Set, Tuple

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / "lib").exists() and (ROOT.parent / "lib").exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import format_submissions, clean_json_response
try:
    from lib.prompt_strategies import build_curriculum_aware_prompt_v2, KC_TAGS
except ModuleNotFoundError:
    from lib.prompts import build_curriculum_aware_prompt_v2, KC_TAGS
from utils.dataset import load_topics_json, load_problem_descriptions

MODEL_ID = "gemini-2.5-flash"
SLEEP_SECONDS = 1.5

client = create_client()
print(f"Working directory: {os.getcwd()}")
print(f"Loaded client for model: {MODEL_ID}")

Working directory: /mnt/d/Projects/kintsugi
Loaded client for model: gemini-2.5-flash


In [2]:
# Cell 1: Load the 3 enriched LLM annotation files to get the exact student-problem pairs
ENRICHED_FILES = [
    "results/human_validation/llm_annotations_10155.json",
    "results/human_validation/llm_annotations_14475.json",
    "results/human_validation/llm_annotations_14476.json"
]

target_students = {}  # student_id -> list of problem_ids (str)

for fpath in ENRICHED_FILES:
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    sid = data.get("student_id", data.get("studentId"))
    pids = list(data.get("annotations", {}).keys())
    target_students[str(sid)] = pids
    print(f"Loaded {len(pids)} target problems for student {sid}")

Loaded 46 target problems for student 10155
Loaded 50 target problems for student 14475
Loaded 50 target problems for student 14476


In [3]:
# Cell 2: Extract code and scores from the dataset for exactly those pairs
best_attempts_df = load_best_attempts_df()

student_submissions = {}

for sid_str, pids in target_students.items():
    sid_int = int(sid_str)
    pid_ints = [int(p) for p in pids]
    
    student_df = best_attempts_df[
        (best_attempts_df["SubjectID"] == sid_int) & 
        (best_attempts_df["ProblemID"].isin(pid_ints))
    ].copy()
    
    # Take latest attempt per problem if there are duplicates
    student_df = student_df.drop_duplicates(subset=["ProblemID"], keep="last")
    student_submissions[sid_str] = student_df
    
    print(f"Found {len(student_df)} submissions for student {sid_str} (expected {len(pids)})")

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Found 46 submissions for student 10155 (expected 46)
Found 50 submissions for student 14475 (expected 50)
Found 50 submissions for student 14476 (expected 50)


In [4]:
# Common prompts & logic setup
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}

EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KC_SET = set(EXACT_KC_TAGS)

def extract_kc_tags(output_obj) -> tuple[list[str], list[str]]:
    """Extract KC tags from Curriculum-Aware LLM output structure."""
    if not isinstance(output_obj, dict):
        return [], []

    valid_tags = set()
    invalid_tags = set()
    
    # 1) Try extracting from list of analyses
    analysis_list = output_obj.get("student_analysis", [])
    if isinstance(analysis_list, dict):
        analysis_list = [analysis_list]
    elif not isinstance(analysis_list, list):
        analysis_list = []

    for analysis in analysis_list:
        if not isinstance(analysis, dict): continue
        for gap in analysis.get("knowledge_gaps", []):
            tag = gap.get("missing_concept", "") if isinstance(gap, dict) else gap
            tag = str(tag).strip()
            if tag:
                if tag in VALID_KC_SET: valid_tags.add(tag)
                else: invalid_tags.add(tag)

        for pred in analysis.get("future_predictions", []):
            tag = pred.get("at_risk_topic", "") if isinstance(pred, dict) else pred
            tag = str(tag).strip()
            if tag:
                if tag in VALID_KC_SET: valid_tags.add(tag)
                else: invalid_tags.add(tag)

    # 2) Fallback to flat structure
    for key in ["knowledge_gaps", "future_predictions"]:
        val = output_obj.get(key, [])
        if isinstance(val, list):
            for x in val:
                tag = x.get("missing_concept", "") if isinstance(x, dict) else (x.get("at_risk_topic", "") if isinstance(x, dict) else x)
                tag = str(tag).strip()
                if tag:
                    if tag in VALID_KC_SET: valid_tags.add(tag)
                    else: invalid_tags.add(tag)
                    
    return sorted(list(valid_tags)), sorted(list(invalid_tags))

def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type='application/json',
            ),
        )
        raw_text = response.text if response and response.text else '{}'
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

In [5]:
# Cell 3: Run Baseline Prompt Strategy (~50 calls total; only ~146 total, done iteratively)
# Depending on ratelimit, you could process sequentially:
import sys

baseline_output_dir = Path("results/human_validation")
baseline_output_dir.mkdir(parents=True, exist_ok=True)

for sid_str, pids in target_students.items():
    output_filepath = baseline_output_dir / f"llm_baseline_annotations_v2_{sid_str}.json"
    
    if output_filepath.exists():
        print(f"Skipping {sid_str}, {output_filepath} already exists.")
        continue
    
    student_df = student_submissions[sid_str]
    annotations = {}
    
    print(f"---- Processing baseline for student {sid_str} ({len(student_df)} problems) ----")
    
    for i, (_, row) in enumerate(student_df.iterrows()):
        pid = str(int(row["ProblemID"]))
        
        # Build prompt tailored minimally
        baseline_prompt = build_curriculum_aware_prompt_v2(
            topics=topics,
            problems=problem_descriptions,
            focus_problem_ids=[int(pid)]
        )
        
        parsed_out, duration, err = run_one_call(row, baseline_prompt)
        sys.stdout.write(".")
        sys.stdout.flush()
        
        if err:
            print(f"\nError on pid {pid}: {err}")
            annotations[pid] = {"gaps": []}
            time.sleep(2 * SLEEP_SECONDS)
        else:
            valid_kcs, _ = extract_kc_tags(parsed_out)
            annotations[pid] = {"gaps": valid_kcs}
            time.sleep(SLEEP_SECONDS)
            
    print(f"\nFinished student {sid_str}.")
    
    # Cell 4 Output format
    output_json = {
        "rater": "LLM_Gemini_Baseline",
        "student_id": sid_str,
        "annotations": annotations
    }
    
    with open(output_filepath, "w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2)

---- Processing baseline for student 10155 (46 problems) ----
..............................................
Finished student 10155.
---- Processing baseline for student 14475 (50 problems) ----
..................................................
Finished student 14475.
---- Processing baseline for student 14476 (50 problems) ----
..................................................
Finished student 14476.


In [6]:
# Cell 5: Load human annotations & LLMs, ready for comparison
def load_annotations(filepath: str) -> Tuple[str, Dict[str, Set[str]]]:
    path = Path(filepath)
    if not path.exists():
        return path.stem, {}
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    rater_name = data.get("rater", path.stem)
    student_id = data.get("student_id", data.get("studentId", "unknown"))
    parsed: Dict[str, Set[str]] = {}
    for pid, val in data.get("annotations", {}).items():
        comp_pid = f"{student_id}_{pid}"
        if isinstance(val, dict) and "gaps" in val:
            gaps = val.get("gaps")
            parsed[comp_pid] = set(gaps) if isinstance(gaps, list) else set()
        elif isinstance(val, list):
            parsed[comp_pid] = set(val)
        else:
            parsed[comp_pid] = set()
    return rater_name, parsed

def merge_rater_files(filepaths: List[str], label: str) -> Tuple[str, Dict[str, Set[str]]]:
    merged = {}
    for fp in filepaths:
        _, anns = load_annotations(fp)
        for comp_pid, kcs in anns.items():
            if comp_pid in merged:
                merged[comp_pid].update(kcs)
            else:
                merged[comp_pid] = set(kcs)
    return label, merged

HUMAN_A_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json",
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json",
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json",
]
HUMAN_B_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json",
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json",
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json",
]
LLM_BASELINE_FILES = [
    "results/human_validation/llm_baseline_annotations_v2_10155.json",
    "results/human_validation/llm_baseline_annotations_v2_14475.json",
    "results/human_validation/llm_baseline_annotations_v2_14476.json"
]

_, anns_ha = merge_rater_files(HUMAN_A_FILES, "Human A")
_, anns_hb = merge_rater_files(HUMAN_B_FILES, "Human B")
_, anns_llm_e = merge_rater_files(ENRICHED_FILES, "LLM Enriched")
_, anns_llm_b = merge_rater_files(LLM_BASELINE_FILES, "LLM Baseline")

common_pids = sorted(set(anns_ha.keys()) & set(anns_hb.keys()) & set(anns_llm_e.keys()) & set(anns_llm_b.keys()))
print(f"Total overlapping problems to evaluate: {len(common_pids)}")

Total overlapping problems to evaluate: 146


In [7]:
# Cell 6: Compute metrics
def compute_metrics(name_a: str, anns_a: Dict[str, Set[str]], name_b: str, anns_b: Dict[str, Set[str]], common_pids: List[str]):
    y_a, y_b = [], []
    for pid in common_pids:
        gaps_a, gaps_b = anns_a.get(pid, set()), anns_b.get(pid, set())
        for kc in EXACT_KC_TAGS:
            y_a.append(1 if kc in gaps_a else 0)
            y_b.append(1 if kc in gaps_b else 0)

    y_a, y_b = np.array(y_a), np.array(y_b)
    n = len(y_a)
    if n == 0: return {"kappa": 0.0, "f1": 0.0, "precision": 0.0, "recall": 0.0}

    po = np.sum(y_a == y_b) / n
    pe = (np.sum(y_a == 1) * np.sum(y_b == 1) + np.sum(y_a == 0) * np.sum(y_b == 0)) / (n * n)
    kappa = (po - pe) / (1 - pe) if (1 - pe) > 1e-12 else 0.0

    tp = int(np.sum((y_a == 1) & (y_b == 1)))
    fp = int(np.sum((y_a == 0) & (y_b == 1)))
    fn = int(np.sum((y_a == 1) & (y_b == 0)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"name": f"{name_a} vs {name_b}", "kappa": kappa, "f1": f1, "precision": precision, "recall": recall}

# Pairwise comparisons
res_h_h = compute_metrics("Human A", anns_ha, "Human B", anns_hb, common_pids)
res_a_base = compute_metrics("Human A", anns_ha, "LLM Baseline", anns_llm_b, common_pids)
res_b_base = compute_metrics("Human B", anns_hb, "LLM Baseline", anns_llm_b, common_pids)
res_a_enr = compute_metrics("Human A", anns_ha, "LLM Enriched", anns_llm_e, common_pids)
res_b_enr = compute_metrics("Human B", anns_hb, "LLM Enriched", anns_llm_e, common_pids)

# Cell 7: Print table
avg_base_kappa = (res_a_base['kappa'] + res_b_base['kappa'])/2
avg_base_f1 = (res_a_base['f1'] + res_b_base['f1'])/2

avg_enr_kappa = (res_a_enr['kappa'] + res_b_enr['kappa'])/2
avg_enr_f1 = (res_a_enr['f1'] + res_b_enr['f1'])/2

print("\n" + "=" * 60)
print(f"{'Pair':<26} {'Kappa':>8} {'F1':>8}")
print("-" * 60)
print(f"{'Human A vs Human B':<26} {res_h_h['kappa']:>8.3f} {res_h_h['f1']:>8.3f}")
print(f"{'Human A vs Enriched LLM':<26} {res_a_enr['kappa']:>8.3f} {res_a_enr['f1']:>8.3f}")
print(f"{'Human B vs Enriched LLM':<26} {res_b_enr['kappa']:>8.3f} {res_b_enr['f1']:>8.3f}")
print(f"{'-- Average vs Enriched LLM':<26} {avg_enr_kappa:>8.3f} {avg_enr_f1:>8.3f}")
print("-" * 60)
print(f"{'Human A vs Baseline LLM':<26} {res_a_base['kappa']:>8.3f} {res_a_base['f1']:>8.3f}")
print(f"{'Human B vs Baseline LLM':<26} {res_b_base['kappa']:>8.3f} {res_b_base['f1']:>8.3f}")
print(f"{'-- Average vs Baseline LLM':<26} {avg_base_kappa:>8.3f} {avg_base_f1:>8.3f}")
print("=" * 60)


Pair                          Kappa       F1
------------------------------------------------------------
Human A vs Human B            0.421    0.459
Human A vs Enriched LLM       0.417    0.448
Human B vs Enriched LLM       0.225    0.271
-- Average vs Enriched LLM    0.321    0.359
------------------------------------------------------------
Human A vs Baseline LLM       0.470    0.502
Human B vs Baseline LLM       0.273    0.322
-- Average vs Baseline LLM    0.371    0.412


In [8]:
# Cell 8: Save overall summary
output_summary = {
    "num_problems": len(common_pids),
    "human_vs_human": res_h_h,
    "average_vs_enriched": {
        "kappa": avg_enr_kappa,
        "f1": avg_enr_f1
    },
    "average_vs_baseline": {
        "kappa": avg_base_kappa,
        "f1": avg_base_f1
    },
    "details": {
        "human_a_vs_baseline": res_a_base,
        "human_b_vs_baseline": res_b_base,
        "human_a_vs_enriched": res_a_enr,
        "human_b_vs_enriched": res_b_enr
    }
}

summary_path = Path("results/human_validation/exp11_baseline_v2_vs_human_metrics.json")
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(output_summary, f, indent=2)
print(f"Saved metrics summary to {summary_path}")

Saved metrics summary to results/human_validation/exp11_baseline_v2_vs_human_metrics.json
